In [13]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import metrics_search_for_two_fragments_df
from clean_and_collect_power_data_alt import Clean
from tqdm import tqdm

In [14]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
test_path = '../../../../data_ds_project/testing_yearly_parquet/'

In [15]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [16]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [17]:
test_path = '../../../../data_ds_project/testing_yearly_parquet/'
# amend as necessary

For testing, just save locally (don't upload it, I think)

In [18]:
test_10 = Clean(10, test_path, systems_cleaned, None, './test_results_10_first/')

In [19]:
my_year = 2016
year_2016_data_basic = test_10.extract_years_data_parquet(my_year)
year_2016_data_standard = test_10.standardize_dataframe(year_2016_data_basic)
year_2016_data_rem = test_10.remove_small_values(year_2016_data_standard)
year_2016_good_days = test_10.keep_good_days_only(year_2016_data_rem)
year_2016_energy_readings = test_10.convert_to_energy_LRS(year_2016_good_days)

Quick loop

In [5]:
for system_id in tqdm(enough_data_parquet_power_list):
    # grab columns from a blank year
    blank_year_pq = pq.ParquetDataset(
        f'{test_path}{system_id}/',
        filters = [('year', '==', 1990)]
    )
    blank_year_df = blank_year_pq.read().to_pandas()
    pow_cols = [col_name for col_name in blank_year_df.columns if ('pow' in col_name)]
    is_inv = any([('inv' in col_name) for col_name in pow_cols])
    is_met = any([('met' in col_name) for col_name in pow_cols])
    if is_inv and is_met:
        met_inv_inputs = ['inverter', 'meter']
        met_inv_names = met_inv_inputs
    elif is_inv:
        met_inv_inputs = ['inverter',]
        met_inv_names = met_inv_inputs
    elif is_met:
        met_inv_inputs = ['meter',]
        met_inv_names = met_inv_inputs
    else:
        met_inv_inputs = [None,]
        met_inv_names = ['unknown',]
    for j in range(len(met_inv_inputs)):
        met_or_inv = met_inv_inputs[j]
        met_or_inv_name = met_inv_names[j]
        system_cleaning_module = Clean(system_id, test_path, systems_cleaned,
                                       met_or_inv, 
                                       f'./test_results/{system_id}/{met_or_inv_name}/')
        system_cleaning_module.clean_all_and_write_to_file()


  0%|          | 0/82 [00:00<?, ?it/s]

100%|██████████| 82/82 [02:30<00:00,  1.83s/it]
